In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from pyprojroot.here import here
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import pickle

In [5]:
ROOT = here()
DATA_DIR = ROOT / "processed_output"
MODEL_DIR = ROOT / "models"
print("Saving to project root:", ROOT)

Saving to project root: c:\Users\adeto\OneDrive\Dokumente\HealthCare 1C\Healthcare-1C-predicting-organ-transplant-waitlist-outcomes-and-optimizing-allocation


In [6]:
X_processed = pd.read_csv(DATA_DIR / "X_processed.csv", index_col=0)
y_frame = pd.read_csv(DATA_DIR / "y_processed.csv", index_col=0)
if y_frame.shape[1] != 1 or not X_processed.index.equals(y_frame.index):
    raise ValueError("Expected one target column aligned with the feature row index.")
if not X_processed.index.is_unique or not X_processed.columns.is_unique:
    raise ValueError("Feature row indices and column names must be unique.")
y_processed = y_frame.iloc[:, 0]
if set(y_processed.unique()) != {0, 1}:
    raise ValueError("Expected a binary 0/1 target without missing values.")
if not all(pd.api.types.is_numeric_dtype(t) for t in X_processed.dtypes):
    raise ValueError("Expected numeric processed features.")
if not np.isfinite(X_processed.to_numpy()).all():
    raise ValueError("Features must not contain missing or infinite values.")

In [ ]:
# Baseline B retains all processed features, including demographics.
excluded_columns = []
print(f"Baseline B uses all {X_processed.shape[1]} features.")

Excluded demographic columns: ['ETHCAT_1', 'ETHCAT_2', 'ETHCAT_4', 'ETHCAT_5', 'ETHCAT_6', 'ETHCAT_7', 'ETHCAT_9', 'ETHCAT_998', 'GENDER_F', 'GENDER_M', 'REGION_1', 'REGION_10', 'REGION_11', 'REGION_2', 'REGION_3', 'REGION_4', 'REGION_5', 'REGION_6', 'REGION_7', 'REGION_8', 'REGION_9']
Baseline A uses 30 features.


In [ ]:
def train_test_RFC(X_train, X_test, y_train, y_test, estimators, maxdepth, min_sample_split, min_sample_leaf, max_features, bootstrap_bool, njobs = -1):
    # Create a logistic regression model
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    model = make_pipeline(StandardScaler(), RandomForestClassifier(
    n_estimators = estimators,       # Number of trees in the forest
    max_depth= maxdepth,           # Restrict depth to prevent overfitting
    min_samples_split = min_sample_split,
    min_samples_leaf = min_sample_leaf, # Smoothing the leaves helps reduce variance in noisy data.
    max_features = max_features,
    bootstrap= bootstrap_bool,            # Use bootstrapping to sample data for each tree
    random_state= 42,        # Ensures reproducible results across production runs
    n_jobs= njobs             # Use all available CPU cores for parallel processing
))

    # Fit the model to the training data
    model.fit(X_train, y_train)

    #Predict the probability of the class of examples of the test data

    proba_predict = model.predict_proba(X_test)

    #Computing the log loss of the probability (proba_predict) by comparing 
    #the actual labels of the test data to the probability predictions

    logloss = log_loss(y_test, proba_predict)

    #Making predictions on the test data, to evaluate the accuracy score
    
    class_label_predict = model.predict(X_test)

    #Computing the accuracy score of the model by comparing the y_test
    #with the class label predictions generated from the step above
    
    accscore = accuracy_score(y_test, class_label_predict)
    
    return {"model": model, "predicted_proba": proba_predict, 'accuracy_score': accscore, 'log_loss': logloss, "predicted_classes": class_label_predict}

In [8]:
def pickling(model, model_name):
    with open(model_name, "wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # Only load pickle files produced here or from another trusted source.
    with open(model_name, "rb") as handle:
        return pickle.load(handle)